In [ ]:
#requirements
!pip install ultralytics opencv-python-headless cvzone
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8m.pt


In [ ]:
#NOTE: This component was not implemented for every single case and we decided on including this component to our project only for the hero example, unlike the other two components.

import cv2
import numpy as np
import math
import cvzone
from ultralytics import YOLO

#Specifying the positional tracking functions in the Tracker class
class Tracker:
    def __init__(self):
        self.center_points = {}
        self.id_count = 0

    def update(self, objects_rect):
        objects_bbs_ids = []
        for rect in objects_rect:
            x, y, w, h = rect
            cx = (x + x + w) // 2
            cy = (y + y + h) // 2

            same_object_detected = False
            for id, pt in self.center_points.items():
                dist = math.hypot(cx - pt[0], cy - pt[1])

                if dist < 35: #max distance constraint
                    self.center_points[id] = (cx, cy)
                    objects_bbs_ids.append([x, y, w, h, id])
                    same_object_detected = True
                    break

            if not same_object_detected:
                self.center_points[self.id_count] = (cx, cy)
                objects_bbs_ids.append([x, y, w, h, self.id_count])
                self.id_count += 1

        new_center_points = {}
        for obj_bb_id in objects_bbs_ids:
            _, _, _, _, object_id = obj_bb_id
            center = self.center_points[object_id]
            new_center_points[object_id] = center

        self.center_points = new_center_points.copy()
        return objects_bbs_ids

#hardcoded example
video_path = '/content/store (1).mp4'
cap = cv2.VideoCapture(video_path)

# Load YOLOv8 medium model
model = YOLO("yolov8m.pt")

# Polygon zones for the current example
area1 = [(213,165),(200,189),(693,373),(697,341)]
area2 = [(195,199),(186,213),(683,404),(689,388)]

tracker = Tracker()
counter1, counter2 = [], []
er, ex = {}, {}

# Video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, 30, (1028, 500))

#Analyzing frame by frame
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (1028, 500))
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = model(frame_rgb, verbose=False)[0]

    detections = []

    #debugger/logger statements
    print("---- New Frame ----")
    for r in results.boxes:
        x1, y1, x2, y2 = map(int, r.xyxy[0])
        cls = int(r.cls[0])
        conf = float(r.conf[0])
        print(f'Detected class {cls} with confidence {conf:.2f}')
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 0), 2)
        cvzone.putTextRect(frame, f'{cls} ({conf:.2f})', (x1, y1 - 10), 1, 1)

        # Add only people to detections (class 0 = person), specifying confidence interval
        if cls == 0 and conf > 0.2:
            w, h = x2 - x1, y2 - y1
            detections.append([x1, y1, w, h])

    print(f"People detected: {len(detections)}")

    bbox_idx = tracker.update(detections)

    #polygon mapping
    for bbox in bbox_idx:
        x1, y1, w, h, id = bbox
        x2, y2 = x1 + w, y1 + h
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

        if cv2.pointPolygonTest(np.array(area1, np.int32), (cx, cy), False) >= 0:
            er[id] = (cx, cy)
        if id in er and cv2.pointPolygonTest(np.array(area2, np.int32), (cx, cy), False) >= 0:
            if id not in counter1:
                counter1.append(id)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cvzone.putTextRect(frame, f'{id}', (cx, cy), 2, 2)
            cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

        if cv2.pointPolygonTest(np.array(area2, np.int32), (cx, cy), False) >= 0:
            ex[id] = (cx, cy)
        if id in ex and cv2.pointPolygonTest(np.array(area1, np.int32), (cx, cy), False) >= 0:
            if id not in counter2:
                counter2.append(id)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cvzone.putTextRect(frame, f'{id}', (cx, cy), 2, 2)
            cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

    cv2.polylines(frame, [np.array(area1, np.int32)], True, (0, 0, 255), 2)
    cv2.polylines(frame, [np.array(area2, np.int32)], True, (0, 0, 255), 2)
    cvzone.putTextRect(frame, f'ENTER: {len(counter1)}', (50, 60), 2, 2)
    cvzone.putTextRect(frame, f'EXIT: {len(counter2)}', (50, 130), 2, 2)

    out.write(frame)

cap.release()
out.release()

# Importing result
from google.colab import files
files.download('output.mp4')


Streaming output truncated to the last 5000 lines.
People detected: 1
---- New Frame ----
Detected class 70 with confidence 0.57
Detected class 68 with confidence 0.50
People detected: 0
---- New Frame ----
Detected class 70 with confidence 0.65
Detected class 68 with confidence 0.52
Detected class 0 with confidence 0.25
People detected: 1
---- New Frame ----
Detected class 70 with confidence 0.64
Detected class 68 with confidence 0.52
Detected class 0 with confidence 0.25
People detected: 1
---- New Frame ----
Detected class 70 with confidence 0.61
Detected class 0 with confidence 0.54
Detected class 68 with confidence 0.51
People detected: 1
---- New Frame ----
Detected class 70 with confidence 0.69
Detected class 68 with confidence 0.55
People detected: 0
---- New Frame ----
Detected class 70 with confidence 0.70
Detected class 0 with confidence 0.55
Detected class 68 with confidence 0.53
People detected: 1
---- New Frame ----
Detected class 70 with confidence 0.64
Detected class 68

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>